# Run Modflow etc. for the current Case.

mf_run.py (or mf_run.ipynb) is used to run the Modflow for the current case.
It sets up the simulation, runs it, and checks for successful termination.

It imports and uses `mf6lab/src/mf_setup.py` to prepare the simulation environment,
and then it writes the simulation files and runs the simulation using the
`Simsim` class from the `mf_setup` module.

The script `mf_setup.py` is general for `mf6lab` and does not have to be changed
for each case, it is mf6lab-wide and, therefore, resides in `mf6lab/src`. It reads local `mf_adapt`, which is specific for the current case. mf_adapt always resides in `mf6lab/Projects/<project>/cases/<case>/src`. The same is true for `mf_analyze` meant to interpret and show the results of the simulation.

`mf_adapt.py` reads the local `settings.py`, which also is specific for the current case. The main use of `mf6lab/Projects/<project>/cases/<case>/src/settings.py` is to set the properties for the current case as a dictionary. But it may also be used to delegate code from `mf_adapt` to minimize clutter in `mf_adapt` and keep it short.

If the simulation does not terminate successfully, it raises an exception.
If the simulation is successful, it prints a success message.
This script can be run from the command line maybe as part of a larger workflow or just from within VScode as usual for any script.

It is assumed that the necessary directories and files are already set up
for the current case. This can be done by importing `mf6tools.Dirs` in mf_adapt and setting `dirs=Dirs()`. It is essential that the `mf_run.py` is launched from within the case directory.

It is also assumed that the necessary Python packages are installed and available in the environment. If not, this will be signaled by errors during imports.

The input files for Modflow will be written out by flopy after mf_adapt was processed. These files are written to the case's subdirectories `SIM` (simulation files), `GWF` (groundwater flow input files), possibly `GWT` if the groundwater transport is to be run and `MF7` if Modpath is invoked. The latter two are irrelevant wihin the `GGOR` project, the (empty) directories still exist in each case's set of subdirectories, and can be ignored.

After having successfully run `mf_run.py` from within `mf6lab/Projects/<project>/cases/<case>/src`, the results of the simulation can be analyzed and presented using the also local script `mf_analyze.py` script.

@ TO 2025-07-02, 2025-11-19

In [ ]:
import os
import sys
import mf6_bootstrap # noqa: F401
from pathlib import Path
from timing import log_timed
from mf6tools import Dirs
from mf_setup import mf_setup
import logging_setup
import logging

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# Doing the actual work and timing the steps

* Runs `mf6lab/src/mf_setup.py` to get the packages and models to be used in the simulation.
* Writes out the Modflow.simulation input files and the Modflow.gwfmodel input files.
* Runs Modflow 6.
* Terminates with a message telling whether or not Modflow 6 ran successfully.

When the run was unsuccessful, inspect the list files of both the simulation (`SIM` directory) and the groundwater flow model (`GWF` directory) to see what went wrong and adapt input accoringly.

If successful, you can run `mf_analyze.py` in the case-directory:
`mf6lab/Projects/<project>/cases/<case>/src/mf_analyze.py`

In [ ]:
dirs = Dirs()
os.chdir(dirs.case)
logging.info("Running from {}".format(os.getcwd()))

with log_timed(logger, "Models and flow packages to be used"):
    # --- running mf_setup
    fp_packages, model_dict, use_models, use_packages = mf_setup()

with log_timed("Getting packages obtained"):
    sim = fp_packages.get('Simsim')

with log_timed(logger, "Smiiulation written"):
    sim.write_simulation(silent=False)

with log_timed(logger, "Simlation ran"):
    success, buff = sim.run_simulation(silent=False)

print('Running success = {}'.format(success))
if not success:
    print(buff)
    logging.critical("Buffer printed because MODFLOW did not terminate normally.")
    raise Exception('MODFLOW did not terminate normally.')
else:
    print("MF6 terminated successfully.")
